In [1]:
print("Hello World")

import os

Hello World


In [5]:
# importing the test data
import urllib.request

os.makedirs("data",exist_ok=True)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url,
"data/input.txt")
print("data has been downloaded")

data has been downloaded


In [7]:
# file handling
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print("length of dataset in characters: ", len(text))
print(text[
    : 1000
])

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hung

In [8]:
# vocabulary creation
# finding the unique elemtnts in it and sorted them out
chars = sorted(list(set(text)))
# this makes a set -> unique and then makes an array and sorts them
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# ### Creating a lookup table, with encode and decode functions

In [9]:
# constructing an an encoder and decoder
stoi = {} # string to index 
itos = {} # index to string
for i,ch in enumerate(chars): # this is us making a dictionary for this to refer later, storing characters with their indexes
    stoi[ch
    ] = i
    itos[i
    ] = ch

# making encode and decode functions
def encode(s):
    encoded_list = []
    for char in s:
        int_val = stoi[char
    ]
        encoded_list.append(int_val)
    return encoded_list

def decode(int_list):
    char_list = []
    for num in int_list:
        char = itos[num
    ]
        char_list.append(char)
    res_string = ''.join(char_list) 
    return res_string

In [15]:
# testing this
encoded_text = encode("Hello I am Mehul")
print(f"Encoded Text: {encoded_text}")
decoded_text = decode(encoded_text)
print(f"Decoded Text: {decoded_text}")

Encoded Text: [20, 43, 50, 50, 53, 1, 21, 1, 39, 51, 1, 25, 43, 46, 59, 50]
Decoded Text: Hello I am Mehul


In [16]:
# much efficient way to write this all
stoi_two = {ch:i for i,ch in enumerate(chars)}
itos_two = {i:ch for i,ch in enumerate(chars)}
encode_two = lambda strs: [stoi_two[c] for c in strs]
decode_two = lambda int_list: "".join([itos_two[int_val] for int_val in int_list])
encoded_text_two = encode_two("Hello I am Mehul")
print(f"Encoded Text: {encoded_text_two}")
decoded_text_two = decode_two(encoded_text_two)
print(f"Decoded Text: {decoded_text_two}")

Encoded Text: [20, 43, 50, 50, 53, 1, 21, 1, 39, 51, 1, 25, 43, 46, 59, 50]
Decoded Text: Hello I am Mehul


NOTE : there are other forms of lookup tables or tokenizers
for eg: Google uses SentencePiece
and openAI has tiktoken

> the better and more vocab dense out tokenizer is: the smaller our token array would be
> this is because we now have more words to fill in
> so smaller sequence -> larger vocabulary
> and larger sequence -> smaller vocablulary 

## Next Up: tokenzie data/input.txt
ok so now we tokenize the Shakespeare text we have
so we will be using PyTorch for this: for the tensor

## Why do we even need a Tensor ?
> tensor: multi dimensional array (0,1,2,3 dimensions)
```python
x = torch.tensor([1, 2, 3])
```
### Why not just use lists ?
- slow, no GPU support and not designed for neural networks

#### Like eg:
- list
    ```python
        result = []
        for i in range(len(a)):
            result.append(a[i] + b[i])
    ```
- tensors (simpler and faster)
    ```python
    a + b
    ```
    > also : ```x = torch.tensor([1,2,3]).to('cuda')``` for GPU support

#### Operations we would do with Tensors
- Embedding lookup
- Matrix multiplication (tensors are regular/rectangular so we can matrix multiply easily/efficiently)
- Softmax
- Backpropagation

#### What does a Tensor look like ?
```python
python_list = [7, 4, 11, 11, 14]
pytorch_tensor = tensor([ 7,  4, 11, 11, 14])
```
```mermaid
text → "hello"
↓
encode → [7, 4, 11, 11, 14]
↓
torch.tensor(...) → tensor([7, 4, 11, 11, 14])
↓
dtype=torch.long → stored as 64-bit integers
```

In [ ]:
# importing pytorch
import torch

# now taking the data and making it a tensor
data = torch.tensor(encode(text),dtype=torch.long) # text: is the text we stored from the data/input.txt
print(data.shape,data.type)
print(f"First 1000 characters: {data[:10000]}")

### Some info
| dtype           | meaning             |
| --------------- | ------------------- |
| `torch.float32` | decimal numbers     |
| `torch.int32`   | integers            |
| `torch.long`    | **64-bit integers** |

- since the data is in IDs we use torch.long
> PyTorch embedding layers require integer indices of type ```torch.long```

- what is shape ? : dimensions of the tensor
```python
encode(text) → [12, 4, 5, 9, 2, ...]
data.shape → (N,)
```
> ```N``` = total number of characters in dataset (~1,000,000)

#### For example:
```python
text = "hello"
encode(text) → [7, 4, 11, 11, 14]
data = torch.tensor([7, 4, 11, 11, 14], dtype=torch.long)
print(data.shape)
```
> output: `(5,)`